In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import time
from pathlib import Path
from typing import Dict, Tuple, Optional, Union, Any, List
import warnings
import scanpy as sc
warnings.filterwarnings('ignore')

# Import additional libraries for neural network training
from sklearn.model_selection import train_test_split, StratifiedKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score
from sklearn.metrics import precision_recall_curve, roc_curve, roc_auc_score, average_precision_score, recall_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Dense, Dropout, BatchNormalization, Input, Add, Activation, 
    MultiHeadAttention, LayerNormalization, Reshape, Flatten,
    GlobalAveragePooling1D, Embedding
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, LearningRateScheduler
from tensorflow.keras.regularizers import l1_l2
from tensorflow.keras import backend as K
from sklearn.ensemble import RandomForestClassifier

# PE + AE TCR

In [2]:
atchley = pd.read_csv("../data/atchley.txt", sep="\t")
# atchley['avg'] = atchley[['f1','f2','f3','f4','f5']] .sum(axis=0)
atchley
index_aa_converter = pd.DataFrame({"aa":atchley['amino.acid'].values, "pos": list(range(20))})

index_aa_converter.index = index_aa_converter['aa'].values
index_aa_converter.pop("aa")
index_aa_converter
atchley.pop('amino.acid')
atchley

cols = atchley.columns
cols
for col in cols:
    atchley[col] =  pd.to_numeric(atchley[col].str.replace("−", "-"), errors='coerce')
atchley['avg'] = atchley.sum(axis=1)
atchley
atchley_aa = pd.concat([index_aa_converter.reset_index(), atchley], axis=1)
atchley_aa
atchley_amino_prop = pd.concat([pd.DataFrame([[0]*atchley_aa.shape[1]],columns=atchley_aa.columns),atchley_aa], axis=0)
atchley_amino_prop
atchley_aa = atchley_amino_prop
atchley_aa.iloc[0,0]='*'
atchley_aa
# word_vectors = np.array(atchley_amino_prop)
word_vectors = np.array(atchley_amino_prop.iloc[:,2:8])
word_vectors.shape

(21, 6)

In [3]:
# We use aa that has length of at most 50
def get_atchley(receptor_seq, length = 35):
    res_atchley = np.zeros((length), dtype='int')
    # print(len(receptor_seq))
    if (len(receptor_seq) <= length):
        for l in range(len(receptor_seq)):
            # print(receptor_seq[l])
            if receptor_seq[l] in index_aa_converter.index:
                res_atchley[l] = 1+index_aa_converter.loc[receptor_seq[l]].values[0]
            else:
                res_atchley[l] = np.random.randint(0,20)
            # res_atchley[l,:] =  atchley.loc[index_aa_converter.loc[receptor_seq[l]].values[0]].values #atchley.loc[receptor_seq[l]]
            # res_atchley = res_atchley.append(atchley.loc[s])
    else:
        for l in range(length):
            # print(receptor_seq[l])
            if receptor_seq[l] in index_aa_converter.index:
                # print(index_aa_converter.loc[receptor_seq[l]].values[0])
                res_atchley[l] = 1+index_aa_converter.loc[receptor_seq[l]].values[0]
            else:
                res_atchley[l] = np.random.randint(0,20)
            # res_atchley[l,:] =  atchley.loc[index_aa_converter.loc[receptor_seq[l]].values[0]].values #atchley.loc[receptor_seq[l]]
            # res_atchley = res_atchley.append(atchley.loc[s])
    return res_atchley

In [4]:
# The input of AAEmbedding is the vectorization of amino acid sequence, i.e., the order of each amino acid letter base in the 
# list of 20 amino acid, we can obtain the input by using get_atchley function
class AAEmbedding(tf.keras.layers.Layer):
    def __init__(self, word_vectors):
        super(AAEmbedding, self).__init__()
        self.word_vectors = tf.constant(word_vectors, dtype=tf.float32)

    def call(self, inputs):
        embedded_inputs = tf.nn.embedding_lookup(self.word_vectors, inputs)
        return embedded_inputs


In [5]:
def positional_encoding(depth, length=25):
  depth = depth
  # print(depth)
  positions = np.arange(length)[:, np.newaxis]   
  positions = np.repeat(positions, depth, axis=1)
  # print(np.repeat(positions, 6, axis=1))
  # print(positions)  # (seq, 1)
  # depths = np.arange(depth)[np.newaxis, :]/depth   # (1, depth)
  # print(depths.shape)
  angle_rates = 1 / (1000)         # (1, depth)
  angle_rads = positions * angle_rates      # (pos, depth)
  # print("Angle radian", angle_rads.shape)
  s = np.sin(angle_rads)[::2]
  c = 1- np.cos(angle_rads)[1::2]
  # print(s)
  # print(c)

  # pos_encoding = np.concatenate(
  #     [np.sin(angle_rads), np.cos(angle_rads)],
  #     axis=-1) 
  pos_encoding = np.vstack(
      [s, c]) 
  return tf.cast(pos_encoding, dtype=tf.float32)



In [6]:
class PositionalEmbedding(tf.keras.layers.Layer):
  def __init__(self, d_model, word_vectors,length = 25):
    super().__init__()
    self.d_model = d_model
    self.length = length
    # self.embedding = tf.keras.layers.Embedding(vocab_size, d_model, mask_zero=True) 
    self.embedding = AAEmbedding(word_vectors)
    self.pos_encoding = positional_encoding(depth=d_model, length=length)

  def compute_mask(self, *args, **kwargs):
    return self.embedding.compute_mask(*args, **kwargs)

  def call(self, x):
    # length = tf.shape(x)[1]

    # print("PositionalEmbedding input shape", x)
    x = self.embedding(x)
    # print(f"beginning x shape {x.shape}")
    # This factor sets the relative scale of the embedding and positonal_encoding.
    x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
    # print('self position encoding value',self.pos_encoding[tf.newaxis, :50, :].shape)
    # print("x value before last line PositionEmbedding",x.shape)
    # scale = tf.cast(x != 0, tf.float32)
    # print(self.pos_encoding[tf.newaxis, :25, :])
    # print(scale.shape)
    # po = self.pos_encoding[tf.newaxis, :25, :]
    # scale = tf.reshape(scale,po.shape)
    # print(po.shape)
    # x = x + tf.multiply(self.pos_encoding[tf.newaxis, :25, :],scale)
    x = x + self.pos_encoding[tf.newaxis, :self.length, :]
    # print("Finish Positional Embedding")
    return x


In [7]:
d_model = 6
pt = PositionalEmbedding(d_model=d_model,word_vectors=word_vectors,length=15)

In [8]:
AA_embed = AAEmbedding(word_vectors)
AA_eem = AA_embed(get_atchley("CASSLGTDTQYF"))
# pt(get_atchley("CASSLGTDTQYF"))

# Donor Split

We just finished doing Autoencoder from PE for TCR

# We do the same for TCR split for AE

In [9]:
# CAP-1 (CEA 571–579), YLSGANLNL
# CAP-1-6D (Modified CAP-1), YLSGADLNL
# CEA-553, QYSWFVNGTF
# CEA-694 (WT), GVLVGVALI
# CEA-694 (L2), GLLVGVALI
# CEA-694 (V9), GLLVGVALV
# CEA-694 (L2V9), GLLVGVALV 
# Identified as a peptide for HLA-A*2402, TYACFVSNL
# CEA.24, LLTFWNPPTTAKLTI
# CEA.33, TAKLTIESTPFNVAE
# CEA.50, EVLLLVHNLPQHLFG

In [23]:
IEDB_tumor_epitope = pd.read_csv("All_epitope_IEDB_McPAS_VDJdb.csv", index_col=0)

In [24]:
IEDB_tumor_epitope

,Epitope
1,SLLMWITQC
2,AAGIGILTV
4,ELAGIGILTV
5,YLEPGPVTV
6,YLEPGAVTA
...,...
40071,QLCDVMFYL
40072,YLYDRLLRV
40073,LYPEFIASI
40074,MLIGIPVYV


In [26]:

train_raw_data = IEDB_tumor_epitope.copy()


In [27]:
train_raw_data[['Epitope']].drop_duplicates()

,Epitope
1,SLLMWITQC
2,AAGIGILTV
4,ELAGIGILTV
5,YLEPGPVTV
6,YLEPGAVTA
...,...
40071,QLCDVMFYL
40072,YLYDRLLRV
40073,LYPEFIASI
40074,MLIGIPVYV


In [28]:

train_peptide = pd.DataFrame()
for s in train_raw_data.Epitope.values:
    train_peptide = pd.concat([train_peptide, pd.DataFrame(pt(get_atchley(s, length=15)).numpy().reshape(1,15*d_model))], axis=0)
# train_TCR.to_csv("train_TCR_PE_only_no_AE_random_split_1.csv", index=False)
# train_TCR.to_csv("train_TCR_PE_only_no_AE_tcr_split_1.csv", index=False)
train_peptide.to_csv(f"train_peptide_PE_only_no_AE_all_peptide_IEDB_McPAS_VDJdb.csv", index=False)
train_peptide = pd.read_csv(f"train_peptide_PE_only_no_AE_all_peptide_IEDB_McPAS_VDJdb.csv")
latent_dim = 50
input_dim = train_peptide.shape[1]
# Encoder
inp = keras.Input(shape=(input_dim,), name='encoder_input')
x = keras.layers.Dense(164)(inp)
x = keras.layers.BatchNormalization()(x)
latent = keras.layers.Dense(latent_dim, name="latent", activation="relu")(x)

# Decoder
x = keras.layers.Dense(164)(latent)
x = keras.layers.BatchNormalization()(x)
out = keras.layers.Dense(input_dim, activation="linear", name='decoder_output')(x) # Linear activation for reconstruction



In [29]:
# Model
model_random_split = keras.Model(inputs=inp, outputs=out, name="peptide_autoencoder")
model_random_split.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
model_random_split.fit(train_peptide, train_peptide, epochs=200, batch_size=64, validation_data=(train_peptide, train_peptide))
model_random_split.save_weights(f'./smart_aligned_v2_peptide_AE_PE_only_no_AE_all_peptide_IEDB_McPAS_VDJdb.h5')
latent_model = keras.Model(model_random_split.input, model_random_split.get_layer("latent").output)
embeddings_peptide_PE = latent_model.predict(train_peptide, batch_size=256)
np.save(f"train_peptide_with_PE_and_AE_embeddings_all_peptide_IEDB_McPAS_VDJdb.npy", embeddings_peptide_PE)

Epoch 1/200
18/18 [==============================] - 0s 5ms/step - loss: 12.7616 - val_loss: 9.7629
Epoch 2/200
18/18 [==============================] - 0s 2ms/step - loss: 8.8053 - val_loss: 7.1551
Epoch 3/200
18/18 [==============================] - 0s 2ms/step - loss: 6.7241 - val_loss: 5.8131
Epoch 4/200
18/18 [==============================] - 0s 2ms/step - loss: 5.3617 - val_loss: 4.8298
Epoch 5/200
18/18 [==============================] - 0s 2ms/step - loss: 4.4074 - val_loss: 4.0727
Epoch 6/200
18/18 [==============================] - 0s 2ms/step - loss: 3.7150 - val_loss: 3.4598
Epoch 7/200
18/18 [==============================] - 0s 2ms/step - loss: 3.2023 - val_loss: 2.9559
Epoch 8/200
18/18 [==============================] - 0s 3ms/step - loss: 2.8498 - val_loss: 2.5538
Epoch 9/200
18/18 [==============================] - 0s 2ms/step - loss: 2.5412 - val_loss: 2.2751
Epoch 10/200
18/18 [==============================] - 0s 2ms/step - loss: 2.3243 - val_loss: 2.0468
Epoch 11